# N091 · 函数图、环与入树

**目标：** 利用每点至多一条出边分离环和链。

**先修：** N082, N083, N088。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 入度剥离；访问时间戳；环长；入树最长链；二元环特例。

## 从问题到算法

函数图每个点至多一条出边，因此每个连通块最终要么终止，要么进入一个环。找最长环只需区分本次路径和旧路径。会议座位中，长度≥3的环不能再接入链；互相喜欢的二元环两端却各能接一条最长入链，且不同二元环可以拼接。

## 最小实现

**本章接口：** `longest_cycle(edges)`、`maximum_invitations(favorite)`

In [1]:
from collections import deque

def longest_cycle(edges):
    visited=[False]*len(edges); answer=-1
    for s in range(len(edges)):
        position={}; u=s
        while u!=-1 and not visited[u]:
            visited[u]=True; position[u]=len(position); u=edges[u]
        if u in position: answer=max(answer,len(position)-position[u])
    return answer

def maximum_invitations(favorite):
    n=len(favorite); degree=[0]*n; depth=[1]*n
    for v in favorite: degree[v]+=1
    q=deque(i for i,d in enumerate(degree) if d==0)
    while q:
        u=q.popleft(); v=favorite[u]; depth[v]=max(depth[v],depth[u]+1); degree[v]-=1
        if degree[v]==0: q.append(v)
    paired=best=0; seen=set()
    for s in range(n):
        if not degree[s] or s in seen: continue
        cycle=[]; u=s
        while u not in seen: seen.add(u); cycle.append(u); u=favorite[u]
        if len(cycle)==2: paired+=sum(depth[v] for v in cycle)
        else: best=max(best,len(cycle))
    return max(paired,best)

## 正确性、前提与复杂度

入度剥离消除所有树枝，depth[v]保留到达v的最长链长；残余顶点恰为环。二元环两端各有一个可用邻座，因此可以接链，其余环每个点的两个邻座均已占满。取所有二元环链之和与单个长环最大值，即覆盖所有可行结构。

**代价：** O(n)时间与空间。longest_cycle允许−1和教学自环；maximum_invitations要求n≥2且favorite[i]≠i，符合本章座位契约。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

favorite=[1,0,1,2,5,4]; print('可邀请人数',maximum_invitations(favorite))
show_table(['员工','喜欢的人'],list(enumerate(favorite)))

可邀请人数 6


员工,喜欢的人
0,1
1,0
2,1
3,2
4,5
5,4


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert longest_cycle([3,3,4,2,3])==3
assert longest_cycle([2,-1,3,1])==-1
assert longest_cycle([0])==1
assert maximum_invitations([1,0,3,2])==4
from itertools import permutations
from random import Random
rng=Random(91)
def brute(f):
    n=len(f); best=0
    for k in range(2,n+1):
        for p in permutations(range(n),k):
            if p[0]!=min(p): continue
            if all(f[p[i]] in (p[i-1],p[(i+1)%k]) for i in range(k)): best=k; break
    return best
for n in range(2,7):
    for _ in range(12):
        f=[rng.choice([j for j in range(n) if j!=i]) for i in range(n)]
        assert maximum_invitations(f)==brute(f)
print('N091: 所有本章断言通过')

N091: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分最大环与多个二元环加链的组合。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 给出普通拓扑不能删除环的解释。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2360. Longest Cycle in a Graph（canonical）
- 2127. Maximum Employees to Be Invited to a Meeting（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。